In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.support_tickets"
)

customers_df = (
    spark.table("customer360_dev.silver.customers")
    .select("customer_id")
)

orders_df = (
    spark.table("customer360_dev.silver.orders")
    .select("order_id")
)

In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "created_timestamp_cast",
        F.to_timestamp("created_timestamp")
    )
    .withColumn(
        "first_response_timestamp_cast",
        F.to_timestamp("first_response_timestamp")
    )
    .withColumn(
        "resolved_timestamp_cast",
        F.to_timestamp("resolved_timestamp")
    )
    .withColumn(
        "closed_timestamp_cast",
        F.to_timestamp("closed_timestamp")
    )
    .withColumn(
        "csat_score_cast",
        F.col("customer_satisfaction_score").cast("int")
    )
    .withColumn(
        "ticket_category_clean",
        F.upper(F.trim("ticket_category"))
    )
    .withColumn(
        "priority_clean",
        F.upper(F.trim("ticket_priority"))
    )
    .withColumn(
        "ticket_status_clean",
        F.upper(F.trim("ticket_status"))
    )
    .withColumn(
        "channel_clean",
        F.upper(F.trim("channel"))
    )
    .withColumn(
        "created_at_cast",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "updated_at_cast",
        F.to_timestamp("updated_at")
    )
)

In [0]:
validated_df = (
    typed_df
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
    .join(
        orders_df.withColumn(
            "_order_exists",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
)

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_ticket_id",
        F.col("ticket_id").isNull()
        | (F.trim(F.col("ticket_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_order_not_found",
        F.col("order_id").isNotNull()
        & F.col("_order_exists").isNull()
    )
    .withColumn(
        "dq_invalid_priority",
        ~F.col("priority_clean").isin(
            "LOW",
            "MEDIUM",
            "HIGH",
            "CRITICAL"
        )
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("ticket_status_clean").isin(
            "OPEN",
            "IN_PROGRESS",
            "RESOLVED",
            "CLOSED",
            "REOPENED"
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_invalid_csat",
    F.col("csat_score_cast").isNotNull()
    & (
        (F.col("csat_score_cast") < 1)
        | (F.col("csat_score_cast") > 5)
    )
)

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_response_before_created",
        F.col("first_response_timestamp_cast").isNotNull()
        & (
            F.col("first_response_timestamp_cast")
            < F.col("created_timestamp_cast")
        )
    )
    .withColumn(
        "dq_resolved_before_created",
        F.col("resolved_timestamp_cast").isNotNull()
        & (
            F.col("resolved_timestamp_cast")
            < F.col("created_timestamp_cast")
        )
    )
    .withColumn(
        "dq_closed_before_resolved",
        F.col("closed_timestamp_cast").isNotNull()
        & F.col("resolved_timestamp_cast").isNotNull()
        & (
            F.col("closed_timestamp_cast")
            < F.col("resolved_timestamp_cast")
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_warning_resolved_without_timestamp",
    (
        (F.col("ticket_status_clean") == "RESOLVED")
        & F.col("resolved_timestamp_cast").isNull()
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_warning_closed_without_timestamp",
    (
        (F.col("ticket_status_clean") == "CLOSED")
        & F.col("closed_timestamp_cast").isNull()
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_ticket_id")
    | F.col("dq_missing_customer_id")
    | F.col("dq_customer_not_found")
    | F.col("dq_order_not_found")
    | F.col("dq_invalid_priority")
    | F.col("dq_invalid_status")
    | F.col("dq_invalid_csat")
    | F.col("dq_response_before_created")
    | F.col("dq_resolved_before_created")
    | F.col("dq_closed_before_resolved")
)

In [0]:
reject_df = validated_df.filter(
    F.col("dq_critical_failure")
)

valid_df = validated_df.filter(
    ~F.col("dq_critical_failure")
)

In [0]:
ticket_window = (
    Window
    .partitionBy("ticket_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

deduped_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(ticket_window)
    )
    .filter(F.col("_row_number") == 1)
)

In [0]:
silver_df = deduped_df.select(
    "ticket_id",
    "customer_id",
    "order_id",

    F.col("ticket_category_clean").alias(
        "ticket_category"
    ),

    F.col("priority_clean").alias(
        "priority"
    ),

    F.col("ticket_status_clean").alias(
        "ticket_status"
    ),

    F.col("channel_clean").alias(
        "channel"
    ),

    F.col("assigned_agent_id").alias("agent_id"),

    F.col("created_timestamp_cast").alias(
        "created_timestamp"
    ),

    F.col("first_response_timestamp_cast").alias(
        "first_response_timestamp"
    ),

    F.col("resolved_timestamp_cast").alias(
        "resolved_timestamp"
    ),

    F.col("closed_timestamp_cast").alias(
        "closed_timestamp"
    ),

    F.col("csat_score_cast").alias(
        "csat_score"
    ),

    "resolution_code",

    "dq_warning_resolved_without_timestamp",
    "dq_warning_closed_without_timestamp",

    F.col("created_at_cast").alias(
        "created_at"
    ),

    F.col("updated_at_cast").alias(
        "updated_at"
    ),

    "_source_file",
    "_source_system",
    "_load_type",
    "_ingestion_timestamp",

    F.current_timestamp().alias(
        "_silver_processed_at"
    )
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.silver.support_tickets"
    )
)

In [0]:
quarantine_df = (
    reject_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("dq_missing_ticket_id"),
                F.lit("MISSING_TICKET_ID")
            ),
            F.when(
                F.col("dq_missing_customer_id"),
                F.lit("MISSING_CUSTOMER_ID")
            ),
            F.when(
                F.col("dq_customer_not_found"),
                F.lit("CUSTOMER_NOT_FOUND")
            ),
            F.when(
                F.col("dq_order_not_found"),
                F.lit("ORDER_NOT_FOUND")
            ),
            F.when(
                F.col("dq_invalid_priority"),
                F.lit("INVALID_PRIORITY")
            ),
            F.when(
                F.col("dq_invalid_status"),
                F.lit("INVALID_TICKET_STATUS")
            ),
            F.when(
                F.col("dq_invalid_csat"),
                F.lit("INVALID_CSAT_SCORE")
            ),
            F.when(
                F.col("dq_response_before_created"),
                F.lit("RESPONSE_BEFORE_CREATED")
            ),
            F.when(
                F.col("dq_resolved_before_created"),
                F.lit("RESOLVED_BEFORE_CREATED")
            ),
            F.when(
                F.col("dq_closed_before_resolved"),
                F.lit("CLOSED_BEFORE_RESOLVED")
            )
        )
    )
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
)

In [0]:
if "_pipeline_run_id" not in spark.table("customer360_dev.quarantine.support_ticket_rejects").columns:
    spark.sql("ALTER TABLE customer360_dev.quarantine.support_ticket_rejects ADD COLUMNS (_pipeline_run_id STRING)")
(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.quarantine.support_ticket_rejects"
    )
)

In [0]:
bronze_count = spark.table(
    "customer360_dev.bronze.support_tickets"
).count()

silver_count = spark.table(
    "customer360_dev.silver.support_tickets"
).count()

reject_count = spark.table(
    "customer360_dev.quarantine.support_ticket_rejects"
).count()

print(f"Bronze Tickets : {bronze_count:,}")
print(f"Silver Tickets : {silver_count:,}")
print(f"Rejected       : {reject_count:,}")

Bronze Tickets : 1,500,096
Silver Tickets : 952,329
Rejected       : 547,767
